In [1]:
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
import urllib.request
import requests
import re
import json as JSON
import sqlite3
from sqlite3 import Error
import sys
import time
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

In [2]:
gameDate, startTime, attendance, stadium, gameDur = " " * 5
teams = { 'CHC': 'CHN',
            'CHW': 'CHA',
            'KCR': 'KCA',
            'LAA': 'ANA',
            'LAD': 'LAN',
            'NYM': 'NYN',
            'SDP': 'SDN',
            'SFG': 'SFN',
            'STL': 'SLN',
            'TBR': 'TBA'}

battingNames = {"ATL":"AtlantaBravesbatting",
                    "ARI":"ArizonaDiamondbacksbatting",
                    "BAL":"BaltimoreOriolesbatting",
                    "BOS":"BostonRedSoxbatting",
                    "CHN":"ChicagoCubsbatting",
                    "CHA":"ChicagoWhiteSoxbatting",
                    "CIN":"CincinnatiRedsbatting",
                    "CLE":"ClevelandGuardiansbatting",
                    "COL":"ColoradoRockiesbatting",
                    "DET":"DetroitTigersbatting",
                    "KCA":"KansasCityRoyalsbatting",
                    "HOU":"HoustonAstrosbatting",
                    "LAA":"LosAngelesAngelsbatting",
                    "ANA":"LosAngelesAngelsbatting",
                    "LAN":"LosAngelesDodgersbatting",
                    "MIA":"MiamiMarlinsbatting",
                    "MIL":"MilwaukeeBrewersbatting",
                    "MIN":"MinnesotaTwinsbatting",
                    "NYN":"NewYorkMetsbatting",
                    "NYY":"NewYorkYankeesbatting",
                    "NYA":"NewYorkYankeesbatting",
                    "OAK":"OaklandAthleticsbatting",
                    "PHI":"PhiladelphiaPhilliesbatting",
                    "PIT":"PittsburghPiratesbatting",
                    "SDN":"SanDiegoPadresbatting",
                    "SEA":"SeattleMarinersbatting",
                    "SFN":"SanFranciscoGiantsbatting",
                    "SLN":"StLouisCardinalsbatting",
                    "TBA":"TampaBayRaysbatting",
                    "TEX":"TexasRangersbatting",
                    "TOR":"TorontoBlueJaysbatting",
                    "WAS":"WashingtonNationalsbatting",
                    "WSN":"WashingtonNationalsbatting"}

pitchingNames = {"ATL":"AtlantaBravespitching",
                    "ARI":"ArizonaDiamondbackspitching",
                    "BAL":"BaltimoreOriolespitching",
                    "BOS":"BostonRedSoxpitching",
                    "CHN":"ChicagoCubspitching",
                    "CHA":"ChicagoWhiteSoxpitching",
                    "CIN":"CincinnatiRedspitching",
                    "CLE":"ClevelandGuardianspitching",
                    "COL":"ColoradoRockiespitching",
                    "DET":"DetroitTigerspitching",
                    "KCA":"KansasCityRoyalspitching",
                    "HOU":"HoustonAstrospitching",
                    "LAA":"LosAngelesAngelspitching",
                    "ANA":"LosAngelesAngelspitching",
                    "LAN":"LosAngelesDodgerspitching",
                    "MIA":"MiamiMarlinspitching",
                    "MIL":"MilwaukeeBrewerspitching",
                    "MIN":"MinnesotaTwinspitching",
                    "NYN":"NewYorkMetspitching",
                    "NYY":"NewYorkYankeespitching",
                    "NYA":"NewYorkYankeespitching",
                    "OAK":"OaklandAthleticspitching",
                    "PHI":"PhiladelphiaPhilliespitching",
                    "PIT":"PittsburghPiratespitching",
                    "SDN":"SanDiegoPadrespitching",
                    "SEA":"SeattleMarinerspitching",
                    "SFN":"SanFranciscoGiantspitching",
                    "SLN":"StLouisCardinalspitching",
                    "TBA":"TampaBayRayspitching",
                    "TEX":"TexasRangerspitching",
                    "TOR":"TorontoBlueJayspitching",
                    "WAS":"WashingtonNationalspitching",
                    "WSN":"WashingtonNationalspitching"}

In [3]:
def boxScoreScrape(team,date):
    bat_data = pd.DataFrame()
    pitch_data = pd.DataFrame()
    gameDate = date
    homeTeam=team
    if homeTeam in teams.keys():
        homeTeam = teams[homeTeam]
        
    url = 'https://www.baseball-reference.com/boxes/%s/%s%s0.shtml' % (homeTeam,homeTeam,gameDate)
    print(url)
    
    # hacky way to account for the doubleheader games i saw
    if gameDate == '20220802': 
        url = 'https://www.baseball-reference.com/boxes/%s/%s%s2.shtml' % (homeTeam,homeTeam,gameDate)

    if gameDate == '20230819':
        url = 'https://www.baseball-reference.com/boxes/%s/%s%s1.shtml' % (homeTeam,homeTeam,gameDate)

    session = requests.Session()
    headers = {}
    res = session.get(url, headers=headers)    

    #print(res)
    
    comm = re.compile("<!--|-->")
    soup = BeautifulSoup(comm.sub("", res.text), 'html.parser')
    #print(soup)
    
    
    # dr = webdriver.Chrome()
    # dr.implicitly_wait(5)
    # dr.get(url)
    # ps = dr.page_source
    # dr.close()
    # soup = BeautifulSoup(ps,"html")
   
    # ---------------
    # options = Options()
    # options.add_argument("--headless")  # Run Chrome in headless mode

    # # dr = webdriver.Chrome(options=options)
    # dr.get(url)
    # dr.implicitly_wait(3)
    # try:
    #     # Explicitly wait for an element to be present (adjust the selector)
    #     print()
    #     WebDriverWait(dr, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "scorebox")))
    #     #WebDriverWait(dr, 3) # .until(EC.presence_of_element_located((By.TAG_NAME, "box"))) # Or a more specific element.)
    #     ps = dr.page_source
    #     dr.close()
    # except Exception as e:
    #     print(f"Loading timed out or element not found: {e}")
    #     ps = dr.page_source # Get whatever page source is available.
    #     print(ps)
    #     dr.close()
    #     return
    # soup = BeautifulSoup(ps, "html")

     # ---------------

    #print("soup", soup)
    scorebox = soup.find(attrs = {'class' : 'scorebox'})
    #print("sb", scorebox)
    scores = scorebox.find_all(attrs = {'class' : 'score'})
    awayTeamScore = int(scores[0].getText())
    homeTeamScore = int(scores[1].getText())
    
    try: 
        teamLinks=scorebox.select("a[href*=teams]")
    except:
        print(hteam,gameDate)
        return (None,None)
        
    ateam = (teamLinks[0]['href'].split('/')[2])
    awayTeam=ateam
    if awayTeam in teams.keys():
        awayTeam = teams[awayTeam]

    #################
    # game info #
    #################
        
    gameMetadata = soup.find(attrs = {'class' : 'scorebox_meta'})
    gameMetadataList = gameMetadata.find_all('div')
    gDate = gameMetadataList[0].text
    startTime = gameMetadataList[1].text.split()[2] + ' ' + gameMetadataList[1].text.split()[3]
    attendance = gameMetadataList[2].text.split()[1]
    stadium = gameMetadataList[3].text.split()[1:]
    stadium = ' '.join(str(x) for x in stadium)
    gameDur = gameMetadataList[4].text.split()[2]

    gameInfoList = [gDate, team, homeTeamScore, awayTeamScore, ateam, startTime, attendance, stadium, gameDur]

    # Create the pandas DataFrame
    game_data = pd.DataFrame([gameInfoList],columns=['Date', 'Home', 'H', 'A', 'Away', 'Time', 'Attendance', 'Stadium', 'Duration'])
    game_data['Date'] = pd.to_datetime(game_data['Date'])
        
    #################
    # home batting #
    #################


    homeBattingTable = soup.find_all("table", {"id":"%s" % battingNames[homeTeam]})
    try:
        data_rows = homeBattingTable[0].find_all('tr')
    except:
        print(homeTeam, gameDate)
        return (None,None,None)
    data_header = homeBattingTable[0].find_all('thead')
    data_header = data_header[0].find_all("tr")
    data_header = data_header[0].find_all("th")
    homeBattingRows = [[td.getText() for td in data_rows[i].find_all(['th','td'])]
        for i in range(len(data_rows))
        ]
    homeBattingStats = pd.DataFrame(homeBattingRows)
    header = []
    for i in range(len(homeBattingStats.columns)):
        header.append(data_header[i].getText())
    homeBattingStats.columns = header
    homeBattingStats = homeBattingStats.loc[homeBattingStats[header[0]] != header[0]].reset_index(drop = True)
    homeBattingStats = homeBattingStats[homeBattingStats.PA != ''].reset_index(drop = True)
    
    # adding HR stats
    homeBattingStats.insert(4,'HR',0)
    for i in range (len(homeBattingStats)):
        # Sum HRs for Team Totals row
        if i == len(homeBattingStats)-1:
            homeBattingStats.at[i,'HR']=homeBattingStats['HR'].sum()
        else:
            for detail in homeBattingStats['Details'][i].split(','):
                if 'HR' in detail:
                    if len(detail.split('·')) > 1:
                        homeBattingStats.at[i,'HR']=int(detail.split('·')[0])
                    else:
                        homeBattingStats.at[i,'HR']=1
                    break
    
    homeBattingStats.insert(1,'Pos','N/A')

    for i in range (len(homeBattingStats)-1):
        words = homeBattingStats['Batting'][i].split()
        position = words[-1]
        name = ' '.join(str(x) for x in words[:-1])
    
        homeBattingStats.at[i, 'Pos'] = position
        homeBattingStats.at[i, 'Batting'] = name
        homeBattingStats.at[i,'Team'] = homeTeam
        homeBattingStats.at[i,'Opp'] = awayTeam
        homeBattingStats.at[i,'Date'] = gameDate
        homeBattingStats.at[i,'G'] = 1

    #print(homeBattingStats.columns)
    try:
        homeBattingStats.drop(columns=['BA','OBP','SLG','OPS','Pit','Str','WPA','WPA+','WPA-','cWPA','aLI','acLI','RE24','PO','A'],inplace=True)
    except:
        homeBattingStats.drop(columns=['BA','OBP','SLG','OPS', 'WPA','WPA+','WPA-','cWPA','aLI','acLI','RE24','PO','A'],inplace=True)

    #################
    # away batting #
    #################

    awayBattingTable = soup.find_all("table", {"id":"%s" % battingNames[awayTeam]})
    try:
        data_rows = awayBattingTable[0].find_all('tr')
    except:
        print(homeTeam, awayTeam, gameDate)
        return (None,None,None)
    data_header = awayBattingTable[0].find_all('thead')
    data_header = data_header[0].find_all("tr")
    data_header = data_header[0].find_all("th")
    awayBattingRows = [[td.getText() for td in data_rows[i].find_all(['th','td'])]
        for i in range(len(data_rows))
        ]
    awayBattingStats = pd.DataFrame(awayBattingRows)
    header = []
    for i in range(len(awayBattingStats.columns)):
        header.append(data_header[i].getText())
    awayBattingStats.columns = header
    awayBattingStats = awayBattingStats.loc[awayBattingStats[header[0]] != header[0]].reset_index(drop = True)
    awayBattingStats = awayBattingStats[awayBattingStats.PA != ''].reset_index(drop = True)
    
    # adding HR stats
    awayBattingStats.insert(4,'HR',0)
    for i in range (len(awayBattingStats)):
        # Sum HRs for Team Totals row
        if i == len(awayBattingStats)-1:
            awayBattingStats.at[i,'HR']=awayBattingStats['HR'].sum()
        else:
            for detail in awayBattingStats['Details'][i].split(','):
                if 'HR' in detail:
                    if len(detail.split('·')) > 1:
                        awayBattingStats.at[i,'HR']=int(detail.split('·')[0])
                    else:
                        awayBattingStats.at[i,'HR']=1
                    break
        
    awayBattingStats.insert(1,'Pos','N/A')

    for i in range (len(awayBattingStats)-1):
        words = awayBattingStats['Batting'][i].split()
        position = words[-1]
        name = ' '.join(str(x) for x in words[:-1])
    
        awayBattingStats.at[i, 'Pos'] = position
        awayBattingStats.at[i, 'Batting'] = name
        awayBattingStats.at[i,'Team'] = awayTeam
        awayBattingStats.at[i,'Opp'] = homeTeam
        awayBattingStats.at[i,'Date'] = gameDate
        awayBattingStats.at[i,'G'] = 1

    try:               
        awayBattingStats.drop(columns=['BA','OBP','SLG','OPS','Pit','Str','WPA','WPA+','WPA-','cWPA','aLI','acLI','RE24','PO','A'],inplace=True)
    except:
        awayBattingStats.drop(columns=['BA','OBP','SLG','OPS', 'WPA','WPA+','WPA-','cWPA','aLI','acLI','RE24','PO','A'],inplace=True)

   
    #################
    # home pitching #
    #################

    homePitchingTable = soup.find_all("table", {"id":"%s" % pitchingNames[homeTeam]})
    data_rows = homePitchingTable[0].find_all('tr')
    data_header = homePitchingTable[0].find_all('thead')
    data_header = data_header[0].find_all("tr")
    data_header = data_header[0].find_all("th")
    homePitchingRows = [[td.getText() for td in data_rows[i].find_all(['th','td'])]
        for i in range(len(data_rows))
        ]
    homePitchingStats = pd.DataFrame(homePitchingRows)
    header = []
    for i in range(len(homePitchingStats.columns)):
        header.append(data_header[i].getText())
    homePitchingStats.columns = header
    homePitchingStats = homePitchingStats.loc[homePitchingStats[header[0]] != header[0]]
    try:
        homePitchingStats.drop(columns=['Ctct','StS','StL','GB','FB','LD','Unk','IR','IS','WPA','aLI','cWPA','acLI','RE24'],inplace=True)
    except:
        homePitchingStats.drop(columns=['IR','IS','WPA','aLI','cWPA','acLI','RE24'],inplace=True)
    
    homePitchingStats = homePitchingStats.reset_index(drop = True)
    for i in range (len(homePitchingStats)):
        homePitchingStats.at[i, 'Pitching'] = homePitchingStats['Pitching'][i].split(',')[0]
        homePitchingStats.at[i,'Team'] = homeTeam
        homePitchingStats.at[i,'Opp'] = awayTeam
        homePitchingStats.at[i,'Date'] = gameDate
        homePitchingStats.at[i,'G'] = 1

    #################
    # away pitching #
    #################

    awayPitchingTable = soup.find_all("table", {"id":"%s" % pitchingNames[awayTeam]})
    data_rows = awayPitchingTable[0].find_all('tr')
    data_header = awayPitchingTable[0].find_all('thead')
    data_header = data_header[0].find_all("tr")
    data_header = data_header[0].find_all("th")
    awayPitchingRows = [[td.getText() for td in data_rows[i].find_all(['th','td'])]
        for i in range(len(data_rows))
        ]
    awayPitchingStats = pd.DataFrame(awayPitchingRows)
    header = []
    for i in range(len(awayPitchingStats.columns)):
        header.append(data_header[i].getText())
    awayPitchingStats.columns = header
    awayPitchingStats = awayPitchingStats.loc[awayPitchingStats[header[0]] != header[0]]
    try:
        awayPitchingStats.drop(columns=['Ctct','StS','StL','GB','FB','LD','Unk','IR','IS','WPA','aLI','cWPA','acLI','RE24'],inplace=True)
    except:
        awayPitchingStats.drop(columns=['IR','IS','WPA','aLI','cWPA','acLI','RE24'],inplace=True)
    awayPitchingStats = awayPitchingStats.reset_index(drop = True)
    for i in range (len(awayPitchingStats)):
        awayPitchingStats.at[i, 'Pitching'] = awayPitchingStats['Pitching'][i].split(',')[0]
        awayPitchingStats.at[i,'Team'] = awayTeam
        awayPitchingStats.at[i,'Opp'] = homeTeam
        awayPitchingStats.at[i,'Date'] = gameDate
        awayPitchingStats.at[i,'G'] = 1
        
        
    bat_data = pd.concat([bat_data,homeBattingStats], ignore_index=True)
    bat_data = pd.concat([bat_data,awayBattingStats], ignore_index=True)
    bat_data = bat_data[bat_data.Batting != 'Team Totals']
    bat_data['Date'] = pd.to_datetime(bat_data['Date'])
    
    pitch_data = pd.concat([pitch_data,homePitchingStats], ignore_index=True)
    pitch_data = pd.concat([pitch_data,awayPitchingStats], ignore_index=True)
    pitch_data = pitch_data[pitch_data.Pitching != 'Team Totals']
    pitch_data['Date'] = pd.to_datetime(pitch_data['Date'])
    
    
    return (bat_data, pitch_data, game_data)

In [4]:
def getAllStats(gameList):
    batting_data = pd.DataFrame()
    pitching_data = pd.DataFrame()
    game_data = pd.DataFrame()
    games = gameList
    
    i=1
    for game in games:
        homeTeam = game[0]
        date = game[1]
        gamebatting, gamepitching, gameinfo = boxScoreScrape(homeTeam,date)
        # space requests to stay out of SportsRef jail
        time.sleep(4)
        if not gamebatting.empty:
            batting_data = pd.concat([batting_data,gamebatting], ignore_index=True)
            pitching_data = pd.concat([pitching_data,gamepitching], ignore_index=True)
            game_data = pd.concat([game_data,gameinfo], ignore_index=True)
            print("logged game %i - %s %s" %(i,homeTeam, date))
        else:
            print("issue with %s %s" %(homeTeam, date))
        i+=1
    return (batting_data, pitching_data, game_data)

In [6]:
games = [('PHI','20080823'),
         ('PHI','20100501'),
         ('PHI','20100706'),
         ('PHI','20100822'),
         ('BOS','20110722'),
         ('PHI','20110724'),
         ('CHC','20140506'),
         ('BOS','20140530'),
         ('COL','20170410'),
         ('LAD','20170416'),
         ('SFG','20180424'),
         ('OAK','20180505'),
         ('SFG','20180926'),
         ('OAK','20190328'),
         ('SFG','20190430'),
         ('OAK','20190906'),
         ('WAS','20210530'),
         ('NYA','20210808'),
         ('LAA','20220408'),
         ('SDP','20220524'),
         ('LAA','20220612'),
         ('SDP','20220626'),
         ('SDP','20220802'),
         ('SDP','20220821'),
         ('LAD','20220903'),
         ('SDP','20220907'),
         ('SDP','20220920'),
         ('SDP','20221001'),
         ('SDP','20230703'),
         ('SDP','20230704'),
         ('SDP','20230815'),
         ('SDP','20230819'),
         ('SDP','20230821'),
         ('SDP','20230822'),
         ('SDP','20230924'),
         ('SDP','20240402'),
         ('SDP','20240514'),
         ('SDP','20240710'),
         ('SDP','20240820'),
         ('SDP','20240918'),
         ('SDP','20241008'),
         ('BAL','20250502'),
         ('COL','20250510'),
         ('ARI','20250614'),
         ('SDP','20250712'),
         ('SDP','20250729'),
         ('SDP','20250803'),
         ('SDP','20250820'),
         ('SDP','20250908'),
         ('SDP','20250909'),
         ('SDP','20260605'),
         ('SDP','20260609'),
         ('SDP','20260810')]

len(games)

53

In [7]:
# scrape for box score info for all games in list
# only run when you want to backfill or add to df

# batting_data, pitching_data, game_data = getAllStats(games)

In [8]:
# get info for a new game to add to csv
newgame = [('SDP','20260825')]
batting_data, pitching_data, game_data = getAllStats(newgame)

https://www.baseball-reference.com/boxes/SDN/SDN202608250.shtml
logged game 1 - SDP 20260825


In [9]:
game_data

,Date,Home,H,A,Away,Time,Attendance,Stadium,Duration
0,2026-08-25,SDP,0,1,PIT,6:40 p.m.,"40,709",Petco Park,2:22


In [10]:
batting_data

,Batting,Pos,AB,R,H,HR,RBI,BB,SO,PA,Details,Team,Opp,Date,G
0,Fernando Tatis Jr.,RF,4,0,1,0,0,0,0,4,2B,SDN,PIT,2026-08-25,1.0
1,Jake Cronenworth,2B,4,0,0,0,0,0,1,4,,SDN,PIT,2026-08-25,1.0
2,Manny Machado,3B,4,0,0,0,0,0,1,4,,SDN,PIT,2026-08-25,1.0
3,Ty France,1B,2,0,0,0,0,2,0,4,,SDN,PIT,2026-08-25,1.0
4,Jase Bowen,PR,0,0,0,0,0,0,0,0,SB,SDN,PIT,2026-08-25,1.0
5,Jackson Merrill,CF,4,0,0,0,0,0,2,4,,SDN,PIT,2026-08-25,1.0
6,Luis Rengifo,DH,3,0,0,0,0,0,2,3,,SDN,PIT,2026-08-25,1.0
7,Luis Campusano,C,3,0,0,0,0,0,2,3,,SDN,PIT,2026-08-25,1.0
8,Xander Bogaerts,SS,3,0,1,0,0,0,0,3,,SDN,PIT,2026-08-25,1.0
9,Dustin Harris,LF,1,0,1,0,0,0,0,2,SH,SDN,PIT,2026-08-25,1.0


In [11]:
pitching_data

,Pitching,IP,H,R,ER,BB,SO,HR,ERA,BF,Pit,Str,GSc,Team,Opp,Date,G
0,Michael King,7,5,0,0,2,7,0,3.23,26,98,68,72,SDN,PIT,2026-08-25,1.0
1,AdriÃ¡n MorejÃ³n,1,0,0,0,0,1,0,2.63,3,6,6,,SDN,PIT,2026-08-25,1.0
2,Mason Miller,1,1,1,1,0,1,1,0.96,4,14,9,,SDN,PIT,2026-08-25,1.0
3,Paul Skenes,6,3,0,0,0,4,0,3.79,21,87,55,70,PIT,SDN,2026-08-25,1.0
4,Camilo Doval,1,0,0,0,1,2,0,3.83,4,17,10,,PIT,SDN,2026-08-25,1.0
5,Gregory Soto,1,0,0,0,0,0,0,3.32,3,11,10,,PIT,SDN,2026-08-25,1.0
6,Mason Montgomery,1,0,0,0,1,2,0,3.19,4,24,16,,PIT,SDN,2026-08-25,1.0


In [12]:
all_boxes = pd.read_csv('all_games.csv',index_col=[0])
all_batting = pd.read_csv('all_batting.csv',index_col=[0])
all_pitching = pd.read_csv('all_pitching.csv',index_col=[0])

In [13]:
all_boxes

,Date,Home,H,A,Away,Time,Attendance,Stadium,Duration
0,08/23/08,PHI,9,2,LAD,3:57 p.m.,"45,019",Citizens Bank Park,2:41
1,05/01/10,PHI,10,0,NYM,3:10 p.m.,"45,264",Citizens Bank Park,2:27
2,07/06/10,PHI,3,6,ATL,7:07 p.m.,"44,715",Citizens Bank Park,3:31
3,08/22/10,PHI,6,0,WSN,1:37 p.m.,"44,539",Citizens Bank Park,2:55
4,07/22/11,BOS,7,4,SEA,7:12 p.m.,"38,048",Fenway Park,2:53
5,07/24/11,PHI,5,3,SDP,1:37 p.m.,"45,241",Citizens Bank Park,2:31
6,05/06/14,CHC,1,5,CHW,7:07 p.m.,"34,305",Wrigley Field,3:30
7,05/30/14,BOS,3,2,TBR,7:11 p.m.,"35,820",Fenway Park,3:55
8,04/10/17,COL,3,5,SDP,6:42 p.m.,"20,504",Coors Field,2:53
9,04/16/17,LAD,1,3,ARI,1:11 p.m.,"39,822",Dodger Stadium,3:22


In [14]:
all_batting

,Batting,Pos,AB,R,H,HR,RBI,BB,SO,PA,Details,Team,Opp,Date,G
0,Jimmy Rollins,SS,4,2,1,0,0,0,0,5,HBP,PHI,LAN,08/23/08,1.0
1,Chase Utley,2B,3,3,2,0,0,2,0,5,NaN,PHI,LAN,08/23/08,1.0
2,Pat Burrell,LF,5,3,3,1,5,0,1,5,"HR,2B",PHI,LAN,08/23/08,1.0
3,Rudy Seanez,P,0,0,0,0,0,0,0,0,NaN,PHI,LAN,08/23/08,1.0
4,Ryan Howard,1B,4,0,1,0,2,1,2,5,2B,PHI,LAN,08/23/08,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1201,William Contreras,C,2,1,0,0,0,2,0,4,NaN,MIL,SDN,08/10/26,1.0
1202,Brandon Lockridge,PR,0,0,0,0,0,0,0,0,CS,MIL,SDN,08/10/26,1.0
1203,Christian Yelich,DH,3,1,1,1,2,0,1,3,HR,MIL,SDN,08/10/26,1.0
1204,Luis Lara,RF,3,0,1,0,0,0,0,3,NaN,MIL,SDN,08/10/26,1.0


In [15]:
all_pitching

,Pitching,IP,H,R,ER,BB,SO,HR,ERA,BF,Pit,Str,GSc,Team,Opp,Date,G
0,Cole Hamels,7.0,5,2,2,0,5,1,3.20,26,97,71,64.0,PHI,LAN,08/23/08,1.0
1,Scott Eyre,1.0,0,0,0,0,2,0,5.28,4,15,11,NaN,PHI,LAN,08/23/08,1.0
2,Rudy Seanez,1.0,0,0,0,0,0,0,3.48,3,4,3,NaN,PHI,LAN,08/23/08,1.0
3,Clayton Kershaw,4.0,7,6,6,2,5,1,4.11,22,91,52,27.0,LAN,PHI,08/23/08,1.0
4,Jason Johnson,1.1,2,3,3,1,1,0,3.96,8,28,17,NaN,LAN,PHI,08/23/08,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442,Yuki Matsui,1.0,0,0,0,0,1,0,2.35,3,15,8,NaN,SDN,MIL,08/10/26,1.0
443,AdriÃ¡n MorejÃ³n,1.0,1,0,0,0,1,0,2.84,3,8,8,NaN,SDN,MIL,08/10/26,1.0
444,Mason Miller,1.0,0,0,0,1,1,0,0.89,3,19,10,NaN,SDN,MIL,08/10/26,1.0
445,Logan Henderson,6.1,5,3,3,1,7,2,2.88,24,91,62,57.0,MIL,SDN,08/10/26,1.0


In [16]:
print("Games: ", all_boxes.shape)
print("New Game: ", game_data.shape)
print("Expecting",all_boxes.shape[0]+game_data.shape[0], all_boxes.shape[1])
print()
print("Batting: ", all_batting.shape)
print("New Batting: ", batting_data.shape)
print("Expecting",all_batting.shape[0]+batting_data.shape[0], all_batting.shape[1])
print()
print("Pitching: ", all_pitching.shape)
print("New Pitching: ", pitching_data.shape)
print("Expecting",all_pitching.shape[0]+pitching_data.shape[0], all_pitching.shape[1])

Games:  (53, 9)
New Game:  (1, 9)
Expecting 54 9

Batting:  (1206, 15)
New Batting:  (22, 15)
Expecting 1228 15

Pitching:  (447, 17)
New Pitching:  (7, 17)
Expecting 454 17


In [17]:
updated_box = pd.concat([all_boxes,game_data], ignore_index=True)
updated_batting = pd.concat([all_batting,batting_data], ignore_index=True)
updated_pitching = pd.concat([all_pitching,pitching_data], ignore_index=True)
updated_box['Date'] = pd.to_datetime(updated_box['Date']).dt.strftime('%m/%d/%y')
updated_batting['Date'] = pd.to_datetime(updated_batting['Date']).dt.strftime('%m/%d/%y')
updated_pitching['Date'] = pd.to_datetime(updated_pitching['Date']).dt.strftime('%m/%d/%y')

/var/folders/c8/1c5f7st91vv6_4wrx1z0q8t80000gn/T/ipykernel_81704/3967705559.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  updated_box['Date'] = pd.to_datetime(updated_box['Date']).dt.strftime('%m/%d/%y')
/var/folders/c8/1c5f7st91vv6_4wrx1z0q8t80000gn/T/ipykernel_81704/3967705559.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  updated_batting['Date'] = pd.to_datetime(updated_batting['Date']).dt.strftime('%m/%d/%y')
/var/folders/c8/1c5f7st91vv6_4wrx1z0q8t80000gn/T/ipykernel_81704/3967705559.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  updated_pitching['Date']

In [18]:
updated_box

,Date,Home,H,A,Away,Time,Attendance,Stadium,Duration
0,08/23/08,PHI,9,2,LAD,3:57 p.m.,"45,019",Citizens Bank Park,2:41
1,05/01/10,PHI,10,0,NYM,3:10 p.m.,"45,264",Citizens Bank Park,2:27
2,07/06/10,PHI,3,6,ATL,7:07 p.m.,"44,715",Citizens Bank Park,3:31
3,08/22/10,PHI,6,0,WSN,1:37 p.m.,"44,539",Citizens Bank Park,2:55
4,07/22/11,BOS,7,4,SEA,7:12 p.m.,"38,048",Fenway Park,2:53
5,07/24/11,PHI,5,3,SDP,1:37 p.m.,"45,241",Citizens Bank Park,2:31
6,05/06/14,CHC,1,5,CHW,7:07 p.m.,"34,305",Wrigley Field,3:30
7,05/30/14,BOS,3,2,TBR,7:11 p.m.,"35,820",Fenway Park,3:55
8,04/10/17,COL,3,5,SDP,6:42 p.m.,"20,504",Coors Field,2:53
9,04/16/17,LAD,1,3,ARI,1:11 p.m.,"39,822",Dodger Stadium,3:22


In [19]:
updated_batting

,Batting,Pos,AB,R,H,HR,RBI,BB,SO,PA,Details,Team,Opp,Date,G
0,Jimmy Rollins,SS,4,2,1,0,0,0,0,5,HBP,PHI,LAN,08/23/08,1.0
1,Chase Utley,2B,3,3,2,0,0,2,0,5,NaN,PHI,LAN,08/23/08,1.0
2,Pat Burrell,LF,5,3,3,1,5,0,1,5,"HR,2B",PHI,LAN,08/23/08,1.0
3,Rudy Seanez,P,0,0,0,0,0,0,0,0,NaN,PHI,LAN,08/23/08,1.0
4,Ryan Howard,1B,4,0,1,0,2,1,2,5,2B,PHI,LAN,08/23/08,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223,Billy Cook,RF,0,0,0,0,0,0,0,0,,PIT,SDN,08/25/26,1.0
1224,Jacob Gonzalez,SS,3,0,0,0,0,0,1,3,,PIT,SDN,08/25/26,1.0
1225,Jared Triolo,3B,3,0,1,0,0,0,0,3,,PIT,SDN,08/25/26,1.0
1226,Henry Davis,C,1,0,0,0,0,0,0,2,"HBP,GDP",PIT,SDN,08/25/26,1.0


In [20]:
updated_pitching

,Pitching,IP,H,R,ER,BB,SO,HR,ERA,BF,Pit,Str,GSc,Team,Opp,Date,G
0,Cole Hamels,7.0,5,2,2,0,5,1,3.2,26,97,71,64.0,PHI,LAN,08/23/08,1.0
1,Scott Eyre,1.0,0,0,0,0,2,0,5.28,4,15,11,NaN,PHI,LAN,08/23/08,1.0
2,Rudy Seanez,1.0,0,0,0,0,0,0,3.48,3,4,3,NaN,PHI,LAN,08/23/08,1.0
3,Clayton Kershaw,4.0,7,6,6,2,5,1,4.11,22,91,52,27.0,LAN,PHI,08/23/08,1.0
4,Jason Johnson,1.1,2,3,3,1,1,0,3.96,8,28,17,NaN,LAN,PHI,08/23/08,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,Mason Miller,1,1,1,1,0,1,1,0.96,4,14,9,,SDN,PIT,08/25/26,1.0
450,Paul Skenes,6,3,0,0,0,4,0,3.79,21,87,55,70,PIT,SDN,08/25/26,1.0
451,Camilo Doval,1,0,0,0,1,2,0,3.83,4,17,10,,PIT,SDN,08/25/26,1.0
452,Gregory Soto,1,0,0,0,0,0,0,3.32,3,11,10,,PIT,SDN,08/25/26,1.0


In [21]:
updated_box.to_csv('all_games.csv')
updated_batting.to_csv('all_batting.csv')
updated_pitching.to_csv('all_pitching.csv')